In [1]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
from dash import Dash, html, dcc, Input, Output, State, dash_table
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Initialize the Dash app
app = Dash(__name__, suppress_callback_exceptions=True)
server = app.server

# Load and preprocess data (you can replace this with your actual data loading)
# For demonstration, I'll create sample data structure
def load_data():
    try:
        df = pd.read_csv("../data/processed_data.csv")

        # For demo - creating sample data
        np.random.seed(586)
        n_hospitals = 500

        sample_data = pd.DataFrame({
            'Facility ID': [f'{i:06d}' for i in range(n_hospitals)],
            'Hospital Name': [f'Hospital {i}' for i in range(n_hospitals)],
            'State': np.random.choice(['CA', 'NY', 'TX', 'FL', 'IL', 'PA', 'OH', 'MI'], n_hospitals),
            'Region': np.random.choice(['Northeast', 'Midwest', 'South', 'West'], n_hospitals),
            'Hospital Type': np.random.choice(['Acute Care', 'Critical Access', 'Childrens', 'Teaching'], n_hospitals),
            'Hospital Ownership': np.random.choice(['Government', 'Nonprofit', 'Proprietary'], n_hospitals),
            'Excess Readmission Ratio': np.random.uniform(0.8, 1.5, n_hospitals),
            'Predicted Readmission Rate': np.random.uniform(0.1, 0.3, n_hospitals),
            'Expected Readmission Rate': np.random.uniform(0.1, 0.3, n_hospitals),
            'Tot_Mdcr_Pymt_Amt': np.random.uniform(100000, 10000000, n_hospitals),
            'Tot_Submtd_Cvrd_Chrg': np.random.uniform(150000, 15000000, n_hospitals),
            'Quality_Score': np.random.uniform(20, 100, n_hospitals),
            'Hospital overall rating': np.random.choice([1, 2, 3, 4, 5], n_hospitals),
            'Cost_Ratio_Total_vs_Medicare': np.random.uniform(0.5, 3.0, n_hospitals),
            'HCAHPS Answer Percent': np.random.uniform(10, 40, n_hospitals),
            'Patient Survey Star Rating': np.random.uniform(1, 5, n_hospitals)
        })

        # Add calculated fields
        sample_data['Readmission_Status'] = np.where(
            sample_data['Excess Readmission Ratio'] > 1,
            'Above Expected',
            'Below Expected'
        )

        # Create efficiency score
        sample_data['Efficiency_Score'] = (
            sample_data['Quality_Score'] /
            sample_data['Tot_Mdcr_Pymt_Amt'].clip(lower=1000) * 10000
        )

        return sample_data

    except Exception as e:
        print(f"Error loading data: {e}")
        return pd.DataFrame()

# Load data
df = load_data()

# Define color schemes
colors = {
    'background': '#f8f9fa',
    'text': '#2c3e50',
    'primary': '#3498db',
    'secondary': '#2ecc71',
    'accent': '#e74c3c'
}

# Define layout
app.layout = html.Div([
    # Header
    html.Div([
        html.H1("🏥 Hospital Performance Analytics Dashboard",
                style={'color': colors['text'], 'marginBottom': 20}),
        html.P("Interactive dashboard for analyzing hospital cost, quality, and readmission metrics",
               style={'color': '#7f8c8d', 'fontSize': '1.1rem'})
    ], style={'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '10px',
              'boxShadow': '0 2px 4px rgba(0,0,0,0.1)', 'marginBottom': '20px'}),

    # Main content
    html.Div([
        # Left sidebar - Filters
        html.Div([
            html.H3("🔍 Filters & Controls", style={'color': colors['text'], 'marginBottom': 20}),

            html.Label("Select Region:", style={'fontWeight': 'bold', 'marginTop': '15px'}),
            dcc.Dropdown(
                id='region-filter',
                options=[{'label': 'All Regions', 'value': 'All'}] +
                        [{'label': r, 'value': r} for r in sorted(df['Region'].unique())],
                value='All',
                clearable=False,
                style={'marginBottom': '20px'}
            ),

            html.Label("Select State:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='state-filter',
                options=[{'label': 'All States', 'value': 'All'}] +
                        [{'label': s, 'value': s} for s in sorted(df['State'].unique())],
                value='All',
                clearable=False,
                style={'marginBottom': '20px'}
            ),

            html.Label("Hospital Type:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='type-filter',
                options=[{'label': 'All Types', 'value': 'All'}] +
                        [{'label': t, 'value': t} for t in sorted(df['Hospital Type'].unique())],
                value='All',
                clearable=False,
                style={'marginBottom': '20px'}
            ),

            html.Label("Hospital Ownership:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='ownership-filter',
                options=[{'label': 'All Ownership Types', 'value': 'All'}] +
                        [{'label': o, 'value': o} for o in sorted(df['Hospital Ownership'].unique())],
                value='All',
                clearable=False,
                style={'marginBottom': '20px'}
            ),

            html.Label("Quality Score Range:", style={'fontWeight': 'bold'}),
            dcc.RangeSlider(
                id='quality-slider',
                min=int(df['Quality_Score'].min()),
                max=int(df['Quality_Score'].max()),
                step=5,
                value=[int(df['Quality_Score'].min()), int(df['Quality_Score'].max())],
                marks={i: str(i) for i in range(int(df['Quality_Score'].min()),
                                               int(df['Quality_Score'].max())+1, 20)},
                tooltip={"placement": "bottom", "always_visible": True}
            ),

            html.Hr(style={'margin': '30px 0'}),

            html.H4("🔧 Chart Controls", style={'color': colors['text']}),

            html.Label("Color By:", style={'fontWeight': 'bold', 'marginTop': '10px'}),
            dcc.RadioItems(
                id='color-by',
                options=[
                    {'label': 'Region', 'value': 'Region'},
                    {'label': 'Hospital Type', 'value': 'Hospital Type'},
                    {'label': 'Ownership', 'value': 'Hospital Ownership'},
                    {'label': 'Readmission Status', 'value': 'Readmission_Status'}
                ],
                value='Region',
                labelStyle={'display': 'block', 'marginBottom': '5px'}
            ),

            html.Button('Reset Filters', id='reset-button', n_clicks=0,
                       style={'marginTop': '20px', 'width': '100%',
                              'padding': '10px', 'backgroundColor': colors['primary'],
                              'color': 'white', 'border': 'none', 'borderRadius': '5px',
                              'cursor': 'pointer'})

        ], style={'width': '20%', 'display': 'inline-block', 'verticalAlign': 'top',
                  'padding': '20px', 'backgroundColor': 'white', 'borderRadius': '10px',
                  'boxShadow': '0 2px 4px rgba(0,0,0,0.1)', 'marginRight': '20px'}),

        # Right content area
        html.Div([
            # KPI Cards
            html.Div([
                html.Div([
                    html.H4("Total Hospitals", style={'color': colors['primary'], 'marginBottom': '5px'}),
                    html.H2(id='total-hospitals', style={'color': colors['text']})
                ], style={'flex': 1, 'padding': '15px', 'backgroundColor': 'white',
                         'borderRadius': '10px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)',
                         'marginRight': '10px'}),

                html.Div([
                    html.H4("Avg. Readmission Ratio", style={'color': colors['primary'], 'marginBottom': '5px'}),
                    html.H2(id='avg-readmission', style={'color': colors['text']})
                ], style={'flex': 1, 'padding': '15px', 'backgroundColor': 'white',
                         'borderRadius': '10px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)',
                         'marginRight': '10px'}),

                html.Div([
                    html.H4("Avg. Quality Score", style={'color': colors['primary'], 'marginBottom': '5px'}),
                    html.H2(id='avg-quality', style={'color': colors['text']})
                ], style={'flex': 1, 'padding': '15px', 'backgroundColor': 'white',
                         'borderRadius': '10px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)',
                         'marginRight': '10px'}),

                html.Div([
                    html.H4("Avg. Medicare Payment", style={'color': colors['primary'], 'marginBottom': '5px'}),
                    html.H2(id='avg-payment', style={'color': colors['text']})
                ], style={'flex': 1, 'padding': '15px', 'backgroundColor': 'white',
                         'borderRadius': '10px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'})
            ], style={'display': 'flex', 'marginBottom': '20px'}),

            # Tabs for different visualizations
            dcc.Tabs([
                # Tab 1: Overview
                dcc.Tab(label='📊 Overview', children=[
                    html.Div([
                        # Row 1: Scatter plot and histogram
                        html.Div([
                            html.Div([
                                dcc.Graph(id='scatter-plot')
                            ], style={'width': '65%', 'display': 'inline-block'}),

                            html.Div([
                                dcc.Graph(id='readmission-histogram')
                            ], style={'width': '34%', 'display': 'inline-block', 'marginLeft': '1%'})
                        ], style={'marginBottom': '20px'}),

                        # Row 2: Bar chart and box plot
                        html.Div([
                            html.Div([
                                dcc.Graph(id='regional-bar')
                            ], style={'width': '49%', 'display': 'inline-block'}),

                            html.Div([
                                dcc.Graph(id='type-boxplot')
                            ], style={'width': '49%', 'display': 'inline-block', 'marginLeft': '2%'})
                        ])
                    ], style={'padding': '10px'})
                ]),

                # Tab 2: Cost Analysis
                dcc.Tab(label='💰 Cost Analysis', children=[
                    html.Div([
                        html.Div([
                            dcc.Graph(id='cost-vs-quality')
                        ], style={'width': '65%', 'display': 'inline-block'}),

                        html.Div([
                            dcc.Graph(id='cost-distribution')
                        ], style={'width': '34%', 'display': 'inline-block', 'marginLeft': '1%'})
                    ], style={'marginBottom': '20px'}),

                    html.Div([
                        dcc.Graph(id='cost-efficiency-heatmap')
                    ])
                ]),

                # Tab 3: Quality Metrics
                dcc.Tab(label='⭐ Quality Metrics', children=[
                    html.Div([
                        html.Div([
                            dcc.Graph(id='quality-radar')
                        ], style={'width': '49%', 'display': 'inline-block'}),

                        html.Div([
                            dcc.Graph(id='patient-experience')
                        ], style={'width': '49%', 'display': 'inline-block', 'marginLeft': '2%'})
                    ]),

                    html.Div([
                        dcc.Graph(id='correlation-matrix')
                    ], style={'marginTop': '20px'})
                ]),

                # Tab 4: Data Explorer
                dcc.Tab(label='📋 Data Explorer', children=[
                    html.Div([
                        html.H4("Hospital Data Table", style={'marginBottom': '15px'}),
                        dash_table.DataTable(
                            id='hospital-table',
                            columns=[{"name": i, "id": i} for i in df.columns[:10]],
                            page_size=10,
                            style_table={'overflowX': 'auto'},
                            style_cell={
                                'textAlign': 'left',
                                'padding': '10px',
                                'minWidth': '100px'
                            },
                            style_header={
                                'backgroundColor': colors['primary'],
                                'color': 'white',
                                'fontWeight': 'bold'
                            }
                        ),

                        html.Div([
                            html.Button("Export to CSV", id="export-btn",
                                       style={'marginTop': '20px', 'padding': '10px 20px',
                                              'backgroundColor': colors['secondary'],
                                              'color': 'white', 'border': 'none',
                                              'borderRadius': '5px', 'cursor': 'pointer'})
                        ]),

                        dcc.Download(id="download-data")
                    ], style={'padding': '20px'})
                ])
            ])
        ], style={'width': '78%', 'display': 'inline-block', 'verticalAlign': 'top'})
    ]),

    # Footer
    html.Div([
        html.Hr(),
        html.P("Hospital Analytics Dashboard | Data Source: CMS Hospital Compare | IMSE 586 Project",
               style={'textAlign': 'center', 'color': '#95a5a6', 'marginTop': '20px'})
    ], style={'marginTop': '40px'})
], style={'backgroundColor': colors['background'], 'padding': '20px', 'minHeight': '100vh'})

# Callback to filter data
@app.callback(
    [Output('total-hospitals', 'children'),
     Output('avg-readmission', 'children'),
     Output('avg-quality', 'children'),
     Output('avg-payment', 'children'),
     Output('scatter-plot', 'figure'),
     Output('readmission-histogram', 'figure'),
     Output('regional-bar', 'figure'),
     Output('type-boxplot', 'figure'),
     Output('cost-vs-quality', 'figure'),
     Output('cost-distribution', 'figure'),
     Output('cost-efficiency-heatmap', 'figure'),
     Output('quality-radar', 'figure'),
     Output('patient-experience', 'figure'),
     Output('correlation-matrix', 'figure'),
     Output('hospital-table', 'data')],
    [Input('region-filter', 'value'),
     Input('state-filter', 'value'),
     Input('type-filter', 'value'),
     Input('ownership-filter', 'value'),
     Input('quality-slider', 'value'),
     Input('color-by', 'value'),
     Input('reset-button', 'n_clicks')]
)
def update_dashboard(region, state, hospital_type, ownership, quality_range, color_by, reset_clicks):
    # Filter data based on inputs
    filtered_df = df.copy()

    # Apply filters
    if region != 'All':
        filtered_df = filtered_df[filtered_df['Region'] == region]

    if state != 'All':
        filtered_df = filtered_df[filtered_df['State'] == state]

    if hospital_type != 'All':
        filtered_df = filtered_df[filtered_df['Hospital Type'] == hospital_type]

    if ownership != 'All':
        filtered_df = filtered_df[filtered_df['Hospital Ownership'] == ownership]

    # Apply quality score filter
    filtered_df = filtered_df[
        (filtered_df['Quality_Score'] >= quality_range[0]) &
        (filtered_df['Quality_Score'] <= quality_range[1])
    ]

    # Calculate KPIs
    total_hospitals = len(filtered_df)
    avg_readmission = f"{filtered_df['Excess Readmission Ratio'].mean():.3f}"
    avg_quality = f"{filtered_df['Quality_Score'].mean():.1f}"
    avg_payment = f"${filtered_df['Tot_Mdcr_Pymt_Amt'].mean():,.0f}"

    # 1. Scatter Plot: Cost vs Readmission
    scatter_fig = px.scatter(
        filtered_df,
        x='Tot_Mdcr_Pymt_Amt',
        y='Excess Readmission Ratio',
        color=color_by,
        hover_data=['Hospital Name', 'State', 'Quality_Score'],
        title='Medicare Payments vs Excess Readmission Ratio',
        labels={
            'Tot_Mdcr_Pymt_Amt': 'Medicare Payments ($)',
            'Excess Readmission Ratio': 'Excess Readmission Ratio'
        },
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    scatter_fig.update_layout(
        plot_bgcolor='white',
        #paper_bgcolor='white',
        hovermode='closest'
    )

    # 2. Readmission Histogram
    hist_fig = px.histogram(
        filtered_df,
        x='Excess Readmission Ratio',
        nbins=30,
        title='Distribution of Excess Readmission Ratio',
        color_discrete_sequence=[colors['primary']]
    )
    hist_fig.add_vline(x=1.0, line_dash="dash", line_color="red",
                       annotation_text="Expected Level")
    hist_fig.update_layout(
        plot_bgcolor='white',
        #paper_bgcolor='white'
    )

    # 3. Regional Performance Bar Chart
    if len(filtered_df) > 0:
        regional_avg = filtered_df.groupby('Region').agg({
            'Excess Readmission Ratio': 'mean',
            'Quality_Score': 'mean'
        }).reset_index()

        bar_fig = px.bar(
            regional_avg,
            x='Region',
            y='Excess Readmission Ratio',
            color='Quality_Score',
            title='Average Readmission Ratio by Region',
            color_continuous_scale='Viridis'
        )
    else:
        bar_fig = go.Figure()
        bar_fig.update_layout(title='No data available for selected filters')

    bar_fig.update_layout(
        plot_bgcolor='white',
        #paper_bgcolor='white'
    )

    # 4. Hospital Type Box Plot
    box_fig = px.box(
        filtered_df,
        x='Hospital Type',
        y='Excess Readmission Ratio',
        color='Hospital Type',
        title='Readmission Ratio Distribution by Hospital Type',
        color_discrete_sequence=px.colors.qualitative.Set3
    )
    box_fig.update_layout(
        plot_bgcolor='white',
        #paper_bgcolor='white',
        xaxis_title="",
        showlegend=False
    )

    # 5. Cost vs Quality Scatter
    cost_quality_fig = px.scatter(
        filtered_df,
        x='Tot_Mdcr_Pymt_Amt',
        y='Quality_Score',
        size='Efficiency_Score',
        color=color_by,
        hover_data=['Hospital Name', 'Excess Readmission Ratio'],
        title='Cost vs Quality Analysis',
        labels={
            'Tot_Mdcr_Pymt_Amt': 'Medicare Payments ($)',
            'Quality_Score': 'Quality Score',
            'Efficiency_Score': 'Efficiency'
        },
        size_max=30
    )
    cost_quality_fig.update_layout(
        plot_bgcolor='white',
        #paper_bgcolor='white'
    )

    # 6. Cost Distribution
    cost_dist_fig = px.violin(
        filtered_df,
        y='Tot_Mdcr_Pymt_Amt',
        x='Region',
        box=True,
        points="all",
        title='Medicare Payment Distribution by Region',
        color='Region'
    )
    cost_dist_fig.update_layout(
        plot_bgcolor='white',
        #paper_bgcolor='white',
        yaxis_title="Medicare Payments ($)"
    )

    # 7. Cost Efficiency Heatmap
    if len(filtered_df) > 0:
        # Create bins for heatmap
        filtered_df['Cost_Bin'] = pd.qcut(filtered_df['Tot_Mdcr_Pymt_Amt'], q=5, labels=False)
        filtered_df['Quality_Bin'] = pd.qcut(filtered_df['Quality_Score'], q=5, labels=False)

        heatmap_data = filtered_df.groupby(['Cost_Bin', 'Quality_Bin']).agg({
            'Excess Readmission Ratio': 'mean',
            'Efficiency_Score': 'mean'
        }).reset_index()

        heatmap_fig = px.density_heatmap(
            filtered_df,
            x='Tot_Mdcr_Pymt_Amt',
            y='Quality_Score',
            z='Excess Readmission Ratio',
            title='Cost-Quality-Readmission Heatmap',
            color_continuous_scale='RdBu_r',
            nbinsx=20,
            nbinsy=20
        )
    else:
        heatmap_fig = go.Figure()
        heatmap_fig.update_layout(title='No data available for selected filters')

    heatmap_fig.update_layout(
        plot_bgcolor='white',
        #paper_bgcolor='white',
        xaxis_title="Medicare Payments ($)",
        yaxis_title="Quality Score"
    )

    # 8. Quality Radar Chart
    if len(filtered_df) > 0:
        radar_data = filtered_df.groupby('Region').agg({
            'Quality_Score': 'mean',
            'Patient Survey Star Rating': 'mean',
            'HCAHPS Answer Percent': 'mean',
            'Excess Readmission Ratio': lambda x: 1/x.mean() if x.mean() > 0 else 0
        }).reset_index()

        radar_fig = go.Figure()

        for region in radar_data['Region'].unique():
            region_data = radar_data[radar_data['Region'] == region]
            radar_fig.add_trace(go.Scatterpolar(
                r=[
                    region_data['Quality_Score'].values[0],
                    region_data['Patient Survey Star Rating'].values[0] * 20,  # Scale to similar range
                    region_data['HCAHPS Answer Percent'].values[0],
                    region_data['Excess Readmission Ratio'].values[0] * 50  # Scale
                ],
                theta=['Quality Score', 'Patient Rating', 'HCAHPS %', 'Readmission (inverted)'],
                fill='toself',
                name=region
            ))

        radar_fig.update_layout(
            polar=dict(
                radialaxis=dict(
                    visible=True,
                    range=[0, 100]
                )),
            showlegend=True,
            title='Regional Quality Metrics Comparison'
        )
    else:
        radar_fig = go.Figure()
        radar_fig.update_layout(title='No data available for selected filters')

    # 9. Patient Experience
    if 'Patient Survey Star Rating' in filtered_df.columns:
        patient_fig = px.scatter(
            filtered_df,
            x='Patient Survey Star Rating',
            y='Excess Readmission Ratio',
            color=color_by,
            size='HCAHPS Answer Percent',
            hover_data=['Hospital Name', 'Quality_Score'],
            title='Patient Experience vs Readmission Rates',
            trendline="ols"
        )
    else:
        patient_fig = px.scatter(
            filtered_df,
            x='Quality_Score',
            y='Excess Readmission Ratio',
            color=color_by,
            title='Quality Score vs Readmission Rates',
            trendline="ols"
        )

    patient_fig.update_layout(
        plot_bgcolor='white',
        #paper_bgcolor='white'
    )

    # 10. Correlation Matrix
    numeric_cols = filtered_df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 1:
        corr_matrix = filtered_df[numeric_cols[:8]].corr()  # Limit to first 8 numeric columns

        corr_fig = px.imshow(
            corr_matrix,
            text_auto=True,
            aspect="auto",
            color_continuous_scale='RdBu_r',
            title='Correlation Matrix of Key Metrics',
            zmin=-1,
            zmax=1
        )
    else:
        corr_fig = go.Figure()
        corr_fig.update_layout(title='Insufficient numeric data for correlation matrix')

    corr_fig.update_layout(
        plot_bgcolor='white',
        #paper_bgcolor='white'
    )

    # 11. Table data
    table_data = filtered_df.head(100).to_dict('records')

    return (total_hospitals, avg_readmission, avg_quality, avg_payment,
            scatter_fig, hist_fig, bar_fig, box_fig,
            cost_quality_fig, cost_dist_fig, heatmap_fig,
            radar_fig, patient_fig, corr_fig, table_data)

# Callback for reset button
@app.callback(
    [Output('region-filter', 'value'),
     Output('state-filter', 'value'),
     Output('type-filter', 'value'),
     Output('ownership-filter', 'value'),
     Output('quality-slider', 'value')],
    Input('reset-button', 'n_clicks')
)
def reset_filters(n_clicks):
    if n_clicks > 0:
        return ['All', 'All', 'All', 'All',
                [int(df['Quality_Score'].min()), int(df['Quality_Score'].max())]]
    return ['All', 'All', 'All', 'All',
            [int(df['Quality_Score'].min()), int(df['Quality_Score'].max())]]

# Callback for export button
@app.callback(
    Output("download-data", "data"),
    Input("export-btn", "n_clicks"),
    prevent_initial_call=True
)
def export_data(n_clicks):
    if n_clicks:
        return dcc.send_data_frame(df.to_csv, "hospital_data_export.csv")

# Run the app
if __name__ == '__main__':
    app.run(jupyter_mode = 'external', debug = False)

Dash app running on http://127.0.0.1:8050/
